# cpet.rag — Colab GPU Ingestion PoC

**Purpose:** Bootstrap the GPU ingestion environment on Google Colab, validate the `core/` package,
and run a smoke check against the corpus index. Full parsing → embedding → ingest pipeline is
wired in Phase 1 cards (#3115–#3119).

---

## Before you start

> **Runtime → Change runtime type → GPU (T4 or A100)**
>
> Docling (PDF parsing) and Jina-embeddings-v3 (late chunking) both require a GPU runtime.
> A CPU runtime will work for the smoke check only.

---

## 1. Verify GPU

Confirm that a CUDA-capable GPU is available. If this cell shows "No devices were found"
you need to enable GPU under **Runtime → Change runtime type**.

In [ ]:
!nvidia-smi

## 2. Clone / Mount the Repo & PDF Store

Choose **one** of the two options below (comment out the other).

- **Option A** — `git clone` the `cpet.rag` repo directly into Colab's `/content/` filesystem.
  Use this when you want to work with the latest code from GitHub.
- **Option B** — Mount Google Drive. Use this when PDFs live in Drive under
  `sinico.papers/pdf/` and you want to read them directly without downloading to S3 first.

> You can enable both — clone the repo *and* mount Drive.

In [ ]:
import os

# ── Option A: git clone ──────────────────────────────────────────────────────
# Replace REPO_URL with the actual GitHub URL
REPO_URL = "https://github.com/<YOUR_ORG>/cpet.rag.git"  # TODO: fill in
REPO_BRANCH = "main"

if not os.path.exists("/content/cpet.rag"):
    !git clone --branch {REPO_BRANCH} {REPO_URL} /content/cpet.rag
else:
    !git -C /content/cpet.rag pull

os.chdir("/content/cpet.rag")
print("CWD:", os.getcwd())

# ── Option B: Mount Google Drive ────────────────────────────────────────────
# Uncomment the block below to mount Drive and expose sinico.papers/pdf/
# DRIVE_MOUNT_POINT = "/content/drive"
# PDF_DIR = "/content/drive/MyDrive/sinico.papers/pdf"  # TODO: confirm Drive path
#
# from google.colab import drive
# drive.mount(DRIVE_MOUNT_POINT)
# print("Drive mounted. PDF directory:", PDF_DIR)
# print("PDF count:", len([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]))

## 3. Install Dependencies

Install the project with the `ingestion`, `vectorstore`, and `gpu` extras.

> **Note:** `docling` pulls in large ML models (~2 GB) and `torch` is ~1.5 GB.
> First-run install on a fresh Colab runtime can take 5–10 minutes.
> Colab disk cache persists within a session but resets on reconnect.

Prefer `uv pip install` for speed. Falls back to plain `pip` if `uv` is not available.

In [ ]:
# Install uv if not present (much faster resolver/installer than pip)
import shutil
if not shutil.which("uv"):
    !pip install -q uv

# Install the project with all required extras
!uv pip install -e ".[ingestion,vectorstore,gpu]" --system

# If uv fails for any reason, fall back to pip:
# !pip install -e ".[ingestion,vectorstore,gpu]"

## 4. Configuration

Set required environment variables before importing `core.config`.

- **JINA_API_KEY** — for Jina-embeddings-v3 API (or local inference via `gpu` extra).
- **GEMINI_API_KEY** — Gemini VLM fallback for complex PDF pages Docling cannot parse.
- **LANCEDB_URI** — for PoC use a local path (`/content/lancedb`); in prod use `s3://<bucket>/lancedb`.

> **Security:** Never hard-code real API keys. Use `google.colab.userdata` or the Secrets panel
> (**Key icon** in the left sidebar) to inject secrets safely.

In [ ]:
import os

# ── Recommended: load from Colab Secrets ────────────────────────────────────
# from google.colab import userdata
# os.environ["JINA_API_KEY"]   = userdata.get("JINA_API_KEY")
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# ── Fallback: set inline (replace placeholders, never commit real keys) ──────
os.environ.setdefault("JINA_API_KEY",   "<YOUR_JINA_API_KEY>")    # TODO
os.environ.setdefault("GEMINI_API_KEY", "<YOUR_GEMINI_API_KEY>")  # TODO
os.environ.setdefault("EMBED_MODEL",    "jinaai/jina-embeddings-v3")

# PoC: store LanceDB in Colab writable /content/ directory
os.environ.setdefault("LANCEDB_URI", "/content/lancedb")

# Production: use S3 (fill in bucket name)
# os.environ["LANCEDB_URI"] = "s3://<YOUR_BUCKET>/lancedb"

# AWS credentials (required only when LANCEDB_URI is an s3:// path)
# os.environ["AWS_ACCESS_KEY_ID"]     = "<...>"
# os.environ["AWS_SECRET_ACCESS_KEY"] = "<...>"
# os.environ["AWS_DEFAULT_REGION"]    = "ap-northeast-2"

# Verify settings load correctly
from core.config import settings
print("embed_model :", settings.embed_model)
print("lancedb_uri :", settings.lancedb_uri)
print("jina_api_key:", "***" if settings.jina_api_key else "(not set)")
print("gemini_api  :", "***" if settings.gemini_api_key else "(not set)")

## 5. Smoke Check

Validate that:
1. `core.metadata.load_corpus_index` can read `data/corpus_index.csv` (794 papers expected).
2. `core.vectorstore.LanceDBStore` can be imported without error.

The full **parsing → embedding → ingest** pipeline will be wired here in Phase 1
(cards #3115 Docling parse, #3117 Late Chunking embed, #3118 load, #3119 IngestionPipeline).

In [ ]:
# ── 1. Corpus index ──────────────────────────────────────────────────────────
from core.metadata import load_corpus_index

papers = load_corpus_index("data/corpus_index.csv")
print(f"Corpus index loaded: {len(papers)} papers")
assert len(papers) > 0, "corpus_index.csv is empty or not found"

# Preview first entry
print("Sample:", papers[0])

# ── 2. VectorStore import ────────────────────────────────────────────────────
from core.vectorstore import LanceDBStore
print("LanceDBStore imported successfully:", LanceDBStore)

# ── Placeholder: Phase 1 pipeline will land here ─────────────────────────────
# from ingestion.pipeline import IngestionPipeline  # wired in #3119
# pipeline = IngestionPipeline(papers=papers, store=LanceDBStore(uri=settings.lancedb_uri))
# pipeline.run()  # Docling parse (#3115) + Jina late-chunk embed (#3117) + load (#3118)

print("
Smoke check PASSED — environment is ready for Phase 1 ingestion pipeline.")

## 6. Next Steps — Phase 1 Cards

Once the smoke check passes, implement the full ingestion pipeline via the following Phase 1 cards:

| Card | Title | Notes |
|------|-------|-------|
| **#3115** | Docling parse | PDF → structured markdown per paper. Gemini VLM fallback for complex pages. |
| **#3117** | Late Chunking embed | Jina-embeddings-v3 late chunking over parsed markdown. |
| **#3118** | LanceDB load | Write chunk vectors + metadata into LanceDB@S3. |
| **#3119** | IngestionPipeline | Orchestrated end-to-end pipeline (LlamaIndex). Incremental, retry-safe. |

**Data flow:**
```
Drive/S3 PDFs
    ↓  #3115 Docling
Parsed Markdown
    ↓  #3117 Jina Late Chunking
Chunk Embeddings
    ↓  #3118 LanceDB load
LanceDB@S3  ←──  serving/ API (cpet.rag)
```

Track progress on the [kanban board](https://cyanlunakanban.vercel.app).